# PMCC 研究评估
比较相同 LEAPS 成交记录、固定成交记录下的成本情景及历史时间分段。
仅使用已实现权益和已完成交易，不包含逐日盯市或样本外验证。

In [ ]:
from options_strategies import PmccConfig, evaluate_pmcc
from options_strategies.shared import load_pmcc_data
from backtest_charts import BacktestReport

config = PmccConfig(symbol="SPY", capital=100_000)

## 加载现有缓存数据
不需要新增行情数据源。请按现有回测 Notebook 的方式配置数据缓存访问。

In [ ]:
options, stock = load_pmcc_data(config.symbol, config.start_date, config.end_date, config.expiration_type)
evaluation = evaluate_pmcc(options, stock, config)  # one PMCC engine run
evaluation.summaries

## 完整区间报告
标的股票价格单独展示，因为其估值口径与已实现期权权益不同。

In [ ]:
report = BacktestReport.from_evaluation(evaluation)
report.plot_dashboard()

## 时间分段诊断
收益率以期初账户权益为基准，成本按成交日期归属。交易统计包含各期间平仓交易的完整开平仓成本。

In [ ]:
report.plot_segments(kind="segment")

In [ ]:
report.plot_segments(kind="quarter")

## 共同覆盖区间与压力成本
显式选择展示范围；全部诊断指标均使用所选区间。

In [ ]:
BacktestReport.from_evaluation(evaluation, scope="common", scenario="stress").plot_dashboard()

## 保存与离线重建
每次运行使用新目录，不覆盖已有结果目录。

In [ ]:
from datetime import datetime, UTC
from pathlib import Path
from options_strategies import EvaluationResult

output = Path(".tmp/evaluations") / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
evaluation.save(output)
report.save_html(str(output / "report.html"))
report.export_trades(output / "trades.csv")
restored = EvaluationResult.load(output)
BacktestReport.from_evaluation(restored).plot_segments(kind="year")